In [12]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import requests
import json
import os
import gradio as gr
import finnhub

In [13]:
load_dotenv(override=True)
openai = OpenAI()
finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

In [9]:
pushover_user=os.getenv("PUSHOVER_USER")
pushover_token=os.getenv("PUSHOVER_API_KEY")
pushover_url="https://api.pushover.net/1/messages.json"

In [10]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [38]:
from tinydb import TinyDB, Query
mn_db = TinyDB('4_reConnect_wk1_lab4_market_news.json')
meta_db = TinyDB('4_reConnect_wk1_lab4_metadata.json')

In [40]:
import time

def refresh_latest_market_news():
    latest_unix_timestamp = int(time.time())
    stale_threshold = latest_unix_timestamp - 4 * 60 * 60  # Subtract 4 hours in seconds
    MetaQuery = Query()
    metadata = ''
    metadata = meta_db.search(MetaQuery.type == 'mn_meta')
    print(metadata)
    if(metadata):
        if(metadata[0]["lastNewsUpdate"] > stale_threshold):
            print('Not time to update yet. Next update in 4 hours')
            return
    

    print('updating now')
    latest_news = finnhub_client.general_news('general',min_id=0)
    print(latest_news)
    mn_db.truncate()
    mn_db.insert_multiple(latest_news)
    meta_db.upsert({'type': 'mn_meta', 'lastNewsUpdate': int(time.time())}, MetaQuery.type == 'mn_meta')


refresh_latest_market_news()
#isNewsStale()

[{'type': 'mn_meta', 'lastNewsUpdate': 1771796668}]
Not time to update yet. Next update in 4 hours


In [41]:
push("Hey!")

Push: Hey!


In [ ]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}
